The example data used in this tutorial can be downloaded here: [Tutorial example dataset](https://drive.google.com/file/d/1eKLQajrzK6-0BtVIr_K0_N8KquWSCye9/view?usp=drive_link).

# data processing

In [ ]:
%%bash
scASprofiler-perp  run --sj-dir ./SJ --gtf gencode.v46.annotation.gtf --outdir ./out --samples-ps 1 --sites-ps 20 --sites-thres 10 --samples-thres 1000

Generating input files from the sj directory...
Executing the main filtering pipeline:
Mode: plate (run_filter_pipeline_plate)
processing and merging splicing data...
Filtering to introns associated to 1 and only 1 gene.
done.
executing repeat and initial threshold filter...
Filtering singletons.
Filtering intron groups associated with more than 1 gene.
Filtering singletons.
Filtering intron groups associated with more than 1 gene.


100%|██████████| 62713/62713 [00:59<00:00, 1054.35it/s]


done
executing sites quality filter by threshold
done.
keep the duplicated site starts and ends (re-group)...
Filtering singletons.
Filtering intron groups associated with more than 1 gene.
Filtering singletons.
Filtering intron groups associated with more than 1 gene.
done.
executing sample quality filter...
done.
Filtering singletons.
Filtering intron groups associated with more than 1 gene.
Filtering singletons.
Filtering intron groups associated with more than 1 gene.


100%|██████████| 2523/2523 [00:02<00:00, 1100.74it/s]


Filtering singletons.
Filtering intron groups associated with more than 1 gene.
Filtering singletons.
Filtering intron groups associated with more than 1 gene.
Done.


# imputation based on pre-trained model

In [ ]:
%%bash
scASprofiler-impute train \
  --data-Sj ./out/filter_sj_counts.csv \
  --data-c ./label.txt \
  --outdir out \
  --clusters 2 \
  --n-epochs 1000 \
  --batch-size 8 \
  --drop-prob 0.1 \
  --patience 10 \
  --overwrite \
  --run-impute \
  --name scasp \
  -k 10

  4%|▍         | 645/17000 [02:55<40:56,  6.66it/s, epoch=38, batch=17, d_loss=0.059, g_loss=0.171]   

Early stopping at epoch 52, best val mse=0.00633783


  5%|▌         | 884/17000 [03:40<1:07:06,  4.00it/s, epoch=52, batch=17, d_loss=0.0451, g_loss=0.0748]


Saved best decoder: models/scasp-100-1000-2-dec_best.pt
Imputed matrix saved to: /home/hpw/article_revision/editor_question_eight/scASP-scasp.csv


# quantify

In [ ]:
%%bash
# scASprofiler-quantify --help
scASprofiler-quantify --input-file ./out/scASP-scasp.csv --outdir ./out

AS ratio calculation complete. Results saved to ./out/as_ratio.csv


# clustering

In [ ]:
# read the imputed data and true labels
import pandas as pd
data = pd.read_csv("./out/as_ratio.csv",index_col=0)
label=pd.read_csv("label.txt",header=None)
data=data.iloc[:,3:]

In [ ]:
import numpy as np
import umap

from scipy.stats import entropy
import matplotlib.pyplot as plt
from sklearn import metrics
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import SpectralClustering, KMeans, AgglomerativeClustering



def reduce_dim(mat, manifold_alg, pca_dim,random_state=7):
    pca = PCA(n_components=pca_dim,random_state=random_state)
    embedding = pca.fit_transform(mat).astype(float)
    if manifold_alg == 'umap':
        reducer = umap.UMAP()
    else:
        reducer = TSNE(random_state=random_state)
    coor = reducer.fit_transform(embedding)
    x = coor[:, 0]
    y = coor[:, 1]
    return embedding, x, y


def make_cluster_labels(embedding, cluster_alg, n_cluster):
    if cluster_alg == 'spectral':
        alg = SpectralClustering(n_clusters=n_cluster, affinity='nearest_neighbors', random_state=7)
    elif cluster_alg == 'agglomerative':
        alg = AgglomerativeClustering(n_clusters=n_cluster)
    else:  # kmeans
        alg = KMeans(n_clusters=n_cluster)
    preds = alg.fit(embedding).labels_
    return preds

def normalized_entropy(clusters, labels, base=None):
    n_clusters = len(np.unique(clusters))
    n_labels = len(np.unique(labels))
    
    n = len(clusters)
    
   
    total_entropy = 0
    for cluster in np.unique(clusters):
   
        cluster_indices = clusters == cluster
        cluster_size = np.sum(cluster_indices)
        
        label_distribution = np.zeros(n_labels)
        cluster_labels = labels[cluster_indices]
        for label in np.unique(labels):
            label_distribution[label] = np.sum(cluster_labels == label) / cluster_size
        
       
        cluster_entropy = entropy(label_distribution, base=base)
        total_entropy += (cluster_size / n) * cluster_entropy
    
 
    max_entropy = np.log(n_clusters)
    

    return total_entropy / max_entropy




embedding, x, y=reduce_dim(mat=data.T, manifold_alg="tsne",pca_dim=20)
predicted_result_label=make_cluster_labels(embedding=embedding,cluster_alg="spectral",n_cluster=2)
true_labels = label[0]
predicted_labels = predicted_result_label


label_encoder = LabelEncoder()
true_labels_encoded = label_encoder.fit_transform(true_labels)
predicted_labels_encoded = np.array(predicted_labels)




adjusted_rand_index = metrics.adjusted_rand_score(true_labels_encoded, predicted_labels_encoded)
print("调整后的兰德指数 (Adjusted Rand Index):", adjusted_rand_index)

adjusted_mutual_info = metrics.adjusted_mutual_info_score(true_labels_encoded, predicted_labels_encoded)
print("调整后的互信息 (Adjusted Mutual Information):", adjusted_mutual_info)

normalized_mutual_info = metrics.normalized_mutual_info_score(true_labels_encoded, predicted_labels_encoded)
print("标凈化的互信息 (Normalized Mutual Information):", normalized_mutual_info)


fowlkes_mallows_score = metrics.fowlkes_mallows_score(true_labels_encoded, predicted_labels_encoded)
print("Fowlkes-Mallows指数:", fowlkes_mallows_score)


v_measure = metrics.v_measure_score(true_labels_encoded, predicted_labels_encoded)
print("V-measure:", v_measure)
homogeneity = metrics.homogeneity_score(true_labels_encoded, predicted_labels_encoded)
print("Homogeneity:", homogeneity)


print("Normalized Entropy(归一化熵):", normalized_entropy(predicted_labels_encoded, true_labels_encoded))

/home/hpw/miniconda3/envs/pytorch_revision/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


调整后的兰德指数 (Adjusted Rand Index): 1.0
调整后的互信息 (Adjusted Mutual Information): 1.0
标凈化的互信息 (Normalized Mutual Information): 1.0
Fowlkes-Mallows指数: 1.0
V-measure: 1.0
Homogeneity: 1.0
Normalized Entropy(归一化熵): 0.0
